# V-MD3 Continuous-Motion SAR Acquisition

Records continuous-motion SAR sweeps with the V-MD3 61 GHz FMCW radar mounted on a Galil-driven linear stage. Each sweep moves the stage across the aperture at a constant speed while the radar records, and logs the stage position for every frame.

**Hardware**

- V-MD3 radar on Ethernet (static IP `192.168.100.201`)
- Griffin Motion linear stage on a Galil DMC-30012 controller over USB (COM port)
- Stage scale is 125 nm per count with about 101 mm of travel between limit switches

**Code layout**

The device code lives in `lib/` and normally needs no edits. Each notebook cell holds the settings it uses, right above the call.

- `lib/radar_link.py`: `Radar` class and the network setup
- `lib/galil_stage.py`: `Stage` class (moves, homing, calibration)
- `lib/sar_sweep.py`: sweep recorder, logs, and `.bin` export

**One-time install**

1. Install gclib from Galil's website using the `.exe` installer.
2. In PowerShell, install the Python wrapper:
   `py -m pip install https://www.galil.com/sw/pub/python/gclib-1.0.1-py3-none-any.whl`

Use wrapper version 1.0.1. The newer 2.0 wrapper needs Python 3.12 and has a different API.

**Each session**

1. Close GDK so the COM port is free.
2. Run the imports cell below.
3. Run **1a** (only after plugging in or power cycling the radar), then **1b**.
4. Run **2** to connect and home the stage.

**Each sweep:** set `FILE_NAME` in **Section 3** and run it.

**When finished:** run **Section 4**. Section 5 (export) and the Appendix (calibration) are optional.

In [ ]:
%load_ext autoreload
%autoreload 2

from lib.radar_link import Radar, run_network_setup
from lib.galil_stage import Stage
from lib.sar_sweep import record_sweep, export_long_csv

## 1. Radar

### 1a. Network setup

Only needed after plugging in or power cycling the radar. Opens a UAC prompt, then prints the setup script's checks.

In [ ]:
RADAR_SETUP_SCRIPT = "C:/My Computer/vmd3/setup_radar.ps1"

run_network_setup(RADAR_SETUP_SCRIPT)

### 1b. Connect and measure frame period

Connects, configures the radar, and times 30 frames. Re-run after changing `RADAR_MODE` or `RADAR_STREAMS`.

If it hangs after "Connected to UDP", power cycle the radar, wait about 10 s, then run 1a and 1b again.

In [ ]:
RADAR_IP = "192.168.100.201"
RADAR_MODE = 0   # RSET 0 = 2D, 6 m max range, 4.69 cm range bins
RADAR_STREAMS = ["radc"]   # any of "radc", "rfft", "done"

# Close the previous connection if this cell is re-run
if "radar" in globals():
    radar.close()

radar = Radar(RADAR_IP)
radar.connect(RADAR_MODE, RADAR_STREAMS)
radar.measure_frame_period()

## 2. Stage

Connects to the controller and homes the stage: it drives to the right limit switch, backs off, and sets that point as the start of the sweep. Close GDK first. Re-run if the stage is bumped or power cycled.

These values were measured for this stage (see the Appendix). Only change them if the stage or limit switches change.

In [ ]:
STAGE_PORT = "COM4 --baud 115200 -d"
COUNTS_PER_MM = 8_000   # 125 nm per count
HOME_DIRECTION = +1   # +1 homes to the right end, -1 to the left
HOME_BACKOFF_COUNTS = 1_000   # 0.125 mm off the limit switch
HOME_SPEED_COUNTS = 50_000   # counts/s, about 6 mm/s
HOME_ACCEL_COUNTS = 1_000_000   # counts/s²

# Close the previous connection if this cell is re-run
if "stage" in globals():
    stage.close()

stage = Stage(STAGE_PORT, COUNTS_PER_MM)
stage.connect()
stage.home(HOME_DIRECTION, HOME_BACKOFF_COUNTS, HOME_SPEED_COUNTS, HOME_ACCEL_COUNTS)

## 3. Record a sweep

Set `FILE_NAME` and run. Repeat for each sweep. Repeated names get `_2`, `_3`... instead of overwriting.

The stage returns to the start, waits for the end of a radar frame, then sweeps at constant speed while every frame is recorded. The speed is set so `N_FRAMES` frames span `SWEEP_LENGTH_MM`. Keep the spacing under λ/4 (about 1.23 mm), which means at least 83 frames for 100 mm.

**Saved to `DATA_DIR/SESSION_FOLDER/bins/`:**

- `FILE_NAME.bin` holds the raw radar frames in recording order.
- `FILE_NAME_positions.csv` has one row per frame:
  - `stream`, `stream_frame`: which frame in the `.bin` (e.g. RADC frame 12)
  - `t_s`: arrival time since motion start
  - `galil_counts`: raw encoder position
  - `x_mm`: 0 at the start (right) end, increasing toward the left
  - `x_center_mm`: position from the sweep center, + at the right end, − at the left end
  - `moving`: 1 while the stage was moving

A summary row is added to `DATA_DIR/SESSION_FOLDER/scan_log.csv`.

Use the logged `x_center_mm` positions for SAR processing rather than assuming exact 1 mm spacing. If the summary warns about a dropped-frame gap, re-run the sweep.

In [ ]:
# Output
DATA_DIR = "data"
SESSION_FOLDER = "continuous_motion_test"
FILE_NAME = "test_sweep"

# Sweep
SWEEP_LENGTH_MM = 100.0
N_FRAMES = 101
SWEEP_ACCEL_COUNTS = 1_000_000   # counts/s², reaches sweep speed in under 0.1 s

bin_path = record_sweep(radar, stage, DATA_DIR, SESSION_FOLDER, FILE_NAME,
                        SWEEP_LENGTH_MM, N_FRAMES, SWEEP_ACCEL_COUNTS)

## 4. End session

Stops the stage and closes both connections. Run this before restarting the kernel or closing VS Code so the radar sees a clean disconnect, which helps avoid the connection hang in 1b next time. Safe to run even if nothing is connected.

In [ ]:
radar.close()
stage.close()

## 5. Export to CSV (optional)

Converts a `.bin` into a long-format CSV with one row per (frame, bin, chirp, channel) value and separate `I` and `Q` columns, saved next to the `.bin` with `_long.csv` added. 2D mode only. Doesn't need the radar or stage connected, just the imports cell.

`EXPORT_PATH` defaults to the last sweep recorded in Section 3. For any other file, set it to a path such as `"data/continuous_motion_test/bins/ball_sweep_2.bin"`.

Long format is large. Each RADC or RFFT frame is 32,768 rows, so a 101-frame RADC sweep is about 3.3 million rows.

In [ ]:
EXPORT_PATH = bin_path   # last recorded sweep, or a full path to any .bin
EXPORT_STREAMS = ["radc"]   # "radc" and/or "rfft", only if they were recorded

export_long_csv(EXPORT_PATH, EXPORT_STREAMS)

## Appendix. Stage calibration (one-time)

Only needed when the stage, encoder, or limit switches change. Run Section 2 first so the stage is connected and homed. Both checks return the stage to the start when done.

Last measured on this stage: switch releases after 600 to 800 counts, full travel 808,382 counts (125 nm per count).

### A1. Limit switch release distance

Steps off the home-side switch 100 counts at a time until it releases. Set `HOME_BACKOFF_COUNTS` in Section 2 to the result plus about 200 counts or more. A result of 20,000 means the switch state never changed, so check the switch wiring or direction.

In [ ]:
stage.measure_switch_release()

### A2. Full travel and stage scale

Drives to the far switch and counts the travel between both switches. With about 101 mm of travel, roughly 810,000 counts means 125 nm per count (`COUNTS_PER_MM = 8_000`), and roughly 1,021,000 counts means 100 nm per count (`COUNTS_PER_MM = 10_000`).

In [ ]:
stage.measure_full_travel()